# Comparing TIAToolbox Nucleus Segmentation with IDC Binary Segmentation Masks

<a href="https://colab.research.google.com/github/fedorov/idc-tiatoolbox/blob/main/notebooks/08_comparing_with_idc_segmentations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Overview

IDC hosts the **Pan-Cancer-Nuclei-Seg-DICOM** analysis results, which include both DICOM Microscopy Bulk Simple Annotations (ANN) and **DICOM Segmentation (SEG)** objects for slides across multiple TCGA cancer collections. While the ANN objects contain nucleus centroid/polygon annotations (compared in [Notebook 07](07_comparing_with_idc_annotations.ipynb)), the SEG objects contain **binary pixel-level segmentation masks** of all nuclei.

In this notebook, we:
1. Find DICOM SEG objects in studies that contain slide microscopy (SM) data
2. Download a slide and its corresponding nucleus segmentation mask
3. Parse the DICOM SEG using `highdicom` to extract the binary mask for a region
4. Run TIAToolbox's HoVer-Net on the same region
5. Convert HoVer-Net instance segmentation to a binary mask
6. Compare the two masks visually and with pixel-level metrics (Dice, IoU)

**GPU recommended** for HoVer-Net inference.

## Installation

Run the cell below to install dependencies. **On Colab, the runtime will automatically restart** after installation to pick up the updated numpy version. After the restart, continue from the imports cell below.

In [ ]:
%pip install tiatoolbox idc-index openslide-bin "numcodecs<0.16" highdicom wsidicom shapely scikit-image

# Restart runtime to pick up updated numpy (required on Colab)
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import os
import joblib
import highdicom as hd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import torch
from matplotlib.patches import Polygon as MplPolygon
from PIL import Image
from skimage.draw import polygon as ski_polygon
from skimage.measure import label as ski_label

from idc_index import IDCClient
from tiatoolbox.wsicore.wsireader import WSIReader
from tiatoolbox.models.engine.nucleus_instance_segmentor import NucleusInstanceSegmentor

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    try:
        torch.zeros(1, device="cuda")
    except RuntimeError:
        print("CUDA available but not functional — falling back to CPU")
        device = "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("Note: GPU is recommended. In Colab: Runtime > Change runtime type > T4 GPU")

## 1. Find a Slide with Pan-Cancer Segmentation Masks

The `Pan-Cancer-Nuclei-Seg-DICOM` analysis result provides both DICOM ANN (polygon annotations) and DICOM SEG (binary segmentation masks) for nucleus segmentation. We discover the SEG data by finding segmentations in studies that also contain slide microscopy (SM) series.

In [ ]:
idc_client = IDCClient()
idc_client.fetch_index("sm_index")
idc_client.fetch_index("seg_index")

In [ ]:
# Find studies that contain both SM images and Pan-Cancer SEG objects
studies_with_seg = idc_client.sql_query("""
    SELECT DISTINCT
        i_seg.StudyInstanceUID,
        i_seg.PatientID,
        i_seg.collection_id,
        COUNT(DISTINCT i_seg.SeriesInstanceUID) as seg_series_count
    FROM index i_seg
    WHERE i_seg.analysis_result_id = 'Pan-Cancer-Nuclei-Seg-DICOM'
        AND i_seg.Modality = 'SEG'
        AND i_seg.collection_id = 'tcga_luad'
        AND i_seg.StudyInstanceUID IN (
            SELECT DISTINCT StudyInstanceUID
            FROM index
            WHERE Modality = 'SM' AND collection_id = 'tcga_luad'
        )
    GROUP BY i_seg.StudyInstanceUID, i_seg.PatientID, i_seg.collection_id
    ORDER BY seg_series_count
    LIMIT 10
""")

print(f"Found {len(studies_with_seg)} studies with both SM and Pan-Cancer SEG data")
studies_with_seg

In [ ]:
# Get SEG + source slide details using seg_index.segmented_SeriesInstanceUID
# to identify the exact slide each SEG was derived from
pan_cancer_segs = idc_client.sql_query("""
    SELECT
        seg.SeriesInstanceUID as seg_series_uid,
        seg.segmented_SeriesInstanceUID as slide_series_uid,
        i_seg.PatientID,
        ROUND(i_seg.series_size_MB, 1) as seg_size_mb,
        ROUND(i_slide.series_size_MB, 1) as slide_size_mb,
        s.ObjectiveLensPower,
        s.min_PixelSpacing_2sf as pixel_spacing_mm
    FROM seg_index seg
    JOIN index i_seg ON seg.SeriesInstanceUID = i_seg.SeriesInstanceUID
    JOIN index i_slide ON seg.segmented_SeriesInstanceUID = i_slide.SeriesInstanceUID
    JOIN sm_index s ON seg.segmented_SeriesInstanceUID = s.SeriesInstanceUID
    WHERE i_seg.analysis_result_id = 'Pan-Cancer-Nuclei-Seg-DICOM'
        AND i_seg.collection_id = 'tcga_luad'
        AND s.ObjectiveLensPower >= 20
    ORDER BY i_seg.series_size_MB ASC
    LIMIT 5
""")

print(f"Found {len(pan_cancer_segs)} SEG + slide pairs")
pan_cancer_segs

In [ ]:
# Select the smallest SEG + slide pair
selected = pan_cancer_segs.iloc[0]
slide_series_uid = selected['slide_series_uid']
seg_series_uid = selected['seg_series_uid']

print(f"Selected:")
print(f"  Patient: {selected['PatientID']}")
print(f"  Slide: {slide_series_uid} ({selected['slide_size_mb']} MB)")
print(f"  SEG: {seg_series_uid} ({selected['seg_size_mb']} MB)")
print(f"  Objective: {selected['ObjectiveLensPower']}x")

## 2. Download the Slide and its Segmentation Mask

In [ ]:
download_dir = './slides'
seg_dir = './segmentations'
os.makedirs(download_dir, exist_ok=True)
os.makedirs(seg_dir, exist_ok=True)

# Download the slide
idc_client.download_from_selection(
    downloadDir=download_dir,
    seriesInstanceUID=[slide_series_uid],
    dirTemplate='%SeriesInstanceUID'
)

# Download the SEG object (flat, no directory template)
idc_client.download_from_selection(
    downloadDir=seg_dir,
    seriesInstanceUID=[seg_series_uid],
    dirTemplate=None
)

slide_path = os.path.join(download_dir, slide_series_uid)
seg_files = [f for f in os.listdir(seg_dir) if f.endswith('.dcm')]
print(f"Downloaded slide: {slide_path}")
print(f"Downloaded {len(seg_files)} SEG file(s)")

In [ ]:
# Open slide with TIAToolbox
reader = WSIReader.open(slide_path)

# DICOMWSIReader may not populate objective_power or mpp
info = reader.info
if info.objective_power is None:
    info.objective_power = float(selected['ObjectiveLensPower'])
if info.mpp is None:
    pixel_spacing_um = float(selected['pixel_spacing_mm']) * 1000
    info.mpp = np.array([pixel_spacing_um, pixel_spacing_um])

print(f"Slide: {type(reader).__name__}, dimensions: {info.slide_dimensions}")
print(f"MPP: {info.mpp}")

thumbnail = reader.slide_thumbnail(resolution=1.25, units="power")
plt.figure(figsize=(10, 8))
plt.imshow(thumbnail)
plt.title(f"Slide Thumbnail ({selected['PatientID']})", fontsize=14)
plt.axis('off')
plt.show()

## 3. Parse the DICOM SEG

The Pan-Cancer-Nuclei-Seg-DICOM SEG objects are DICOM Segmentations stored as TILED_FULL binary masks. Each SEG series has one segment ("Nuclei"). Importantly, the SEG may be stored at a **different resolution** than the slide's baseline — we need to compute the scale factor between the two coordinate systems.

In [ ]:
# Load the SEG DICOM object with lazy frame retrieval
seg_path = os.path.join(seg_dir, seg_files[0])
seg = hd.seg.segread(seg_path, lazy_frame_retrieval=True)

print(f"SEG Total Pixel Matrix (w x h): {seg.TotalPixelMatrixColumns} x {seg.TotalPixelMatrixRows}")
print(f"Slide dimensions (w x h): {info.slide_dimensions}")
print(f"Number of frames: {seg.NumberOfFrames}")
print(f"Tile size: {seg.Columns} x {seg.Rows}")
print(f"Segmentation type: {seg.segmentation_type}")
print(f"Number of segments: {len(seg.SegmentSequence)}")
for s_desc in seg.SegmentSequence:
    print(f"  Segment {s_desc.SegmentNumber}: {s_desc.SegmentLabel}")

# Compute scale factor between slide baseline and SEG pixel coordinates
# The SEG may be at a lower resolution than the slide's baseline
seg_pixel_spacing = float(
    seg.SharedFunctionalGroupsSequence[0].PixelMeasuresSequence[0].PixelSpacing[0]
)
slide_pixel_spacing = float(selected['pixel_spacing_mm'])
baseline_to_seg_scale = slide_pixel_spacing / seg_pixel_spacing

print(f"\nSEG pixel spacing: {seg_pixel_spacing:.6f} mm")
print(f"Slide pixel spacing: {slide_pixel_spacing:.6f} mm")
print(f"Scale factor (baseline → SEG): {baseline_to_seg_scale:.4f}")
print(f"  i.e., 1 SEG pixel = {1/baseline_to_seg_scale:.1f} baseline pixels")

## 4. Extract a Region of Interest

We select a tissue-rich region, extract both the H&E tile and the corresponding binary mask from the DICOM SEG.

**Note on resolution:** The SEG may be stored at a lower resolution than the slide's baseline (e.g., ~4x coarser for a 40x slide). When comparing masks, we resize the SEG mask to the tile dimensions using nearest-neighbor interpolation. This resolution difference means pixel-level overlap metrics (Dice, IoU) will be lower than if both masks were at the same native resolution, but nucleus-level counts and spatial density comparisons remain meaningful.

In [ ]:
# Choose a tissue-rich region
gray = np.mean(thumbnail, axis=2)
tissue_mask = gray < 200
tissue_coords = np.argwhere(tissue_mask)
center_y, center_x = tissue_coords.mean(axis=0).astype(int)

slide_w, slide_h = info.slide_dimensions
baseline_x = int(center_x * slide_w / thumbnail.shape[1])
baseline_y = int(center_y * slide_h / thumbnail.shape[0])

# Extract a 2048x2048 tile in baseline coordinates
# DICOMWSIReader has coordinate issues at non-native resolutions,
# so we read at native resolution and resize if needed.
tile_size = 2048
baseline_mpp = float(info.mpp[0])
target_mpp = 0.5

baseline_extent = int(tile_size * target_mpp / baseline_mpp)

bounds = (
    max(0, baseline_x - baseline_extent // 2),
    max(0, baseline_y - baseline_extent // 2),
    min(slide_w, baseline_x + baseline_extent // 2),
    min(slide_h, baseline_y + baseline_extent // 2),
)

tile = reader.read_bounds(
    bounds=bounds,
    resolution=info.objective_power,
    units="power",
)

if tile.shape[0] != tile_size or tile.shape[1] != tile_size:
    tile = np.array(Image.fromarray(tile).resize((tile_size, tile_size), Image.LANCZOS))

print(f"Tile shape: {tile.shape}")
print(f"Baseline bounding box: ({bounds[0]}, {bounds[1]}) to ({bounds[2]}, {bounds[3]})")

In [ ]:
# Extract the binary mask from the DICOM SEG for the same region.
# The SEG may be at a different resolution than the slide baseline,
# so we convert baseline coordinates to SEG coordinates using the scale factor.
# get_total_pixel_matrix uses 1-based indexing with exclusive end (like Python range).
seg_row_start = int(bounds[1] * baseline_to_seg_scale) + 1
seg_row_end = int(bounds[3] * baseline_to_seg_scale) + 1
seg_col_start = int(bounds[0] * baseline_to_seg_scale) + 1
seg_col_end = int(bounds[2] * baseline_to_seg_scale) + 1

# Clamp to SEG dimensions
seg_row_start = max(1, seg_row_start)
seg_row_end = min(seg.TotalPixelMatrixRows + 1, seg_row_end)
seg_col_start = max(1, seg_col_start)
seg_col_end = min(seg.TotalPixelMatrixColumns + 1, seg_col_end)

print(f"Baseline bounds: x=[{bounds[0]}, {bounds[2]}], y=[{bounds[1]}, {bounds[3]}]")
print(f"SEG coordinates: row=[{seg_row_start}, {seg_row_end}), col=[{seg_col_start}, {seg_col_end})")

mask_region = seg.get_total_pixel_matrix(
    row_start=seg_row_start,
    row_end=seg_row_end,
    column_start=seg_col_start,
    column_end=seg_col_end,
    segment_numbers=[1],
    combine_segments=True,
)

print(f"Raw mask shape: {mask_region.shape}")
print(f"Nucleus pixels: {mask_region.sum()} / {mask_region.size} ({100*mask_region.sum()/mask_region.size:.1f}%)")

# Resize mask to match tile dimensions (2048x2048)
# Use NEAREST interpolation to preserve binary values
seg_mask = np.array(Image.fromarray(mask_region.astype(np.uint8)).resize(
    (tile_size, tile_size), Image.NEAREST
))

print(f"Resized mask shape: {seg_mask.shape}")
print(f"Nucleus pixels after resize: {seg_mask.sum()}")

In [ ]:
# Visualize the SEG mask on the tile
fig, axes = plt.subplots(1, 3, figsize=(21, 7))

axes[0].imshow(tile)
axes[0].set_title("H&E Tile", fontsize=13)
axes[0].axis('off')

axes[1].imshow(seg_mask, cmap='gray')
axes[1].set_title(f"IDC SEG Mask ({seg_mask.sum()} nucleus pixels)", fontsize=13)
axes[1].axis('off')

# Overlay: mask in green on the H&E tile
overlay = tile.copy()
overlay[seg_mask > 0] = (overlay[seg_mask > 0] * 0.5 + np.array([0, 255, 0]) * 0.5).astype(np.uint8)
axes[2].imshow(overlay)
axes[2].set_title("SEG Mask Overlay", fontsize=13)
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 5. Run TIAToolbox HoVer-Net on the Same Region

In [ ]:
# Save tile for HoVer-Net
tile_path = './tile_for_seg_comparison.png'
Image.fromarray(tile).save(tile_path)

# Run HoVer-Net
segmentor = NucleusInstanceSegmentor(
    pretrained_model="hovernet_fast-pannuke",
    num_loader_workers=0,
    num_postproc_workers=0,
    batch_size=8,
)

output = segmentor.predict(
    imgs=[tile_path],
    mode="tile",
    save_dir="./seg_comparison_results/",
    resolution=1.0,
    units="baseline",
    device=device,
)

print("HoVer-Net inference complete!")

In [ ]:
# Load HoVer-Net results
hovernet_nuclei = joblib.load(output[0][1] + '.dat')

hovernet_centroids = np.array([nuc['centroid'] for nuc in hovernet_nuclei.values()])
hovernet_types = np.array([nuc['type'] for nuc in hovernet_nuclei.values()])

print(f"TIAToolbox HoVer-Net detected: {len(hovernet_nuclei)} nuclei")

## 6. Convert HoVer-Net Instance Segmentation to Binary Mask

HoVer-Net produces per-nucleus contour polygons. We rasterize these contours into a single binary mask for pixel-level comparison with the IDC SEG mask.

In [ ]:
# Rasterize HoVer-Net contours to a binary mask
hovernet_mask = np.zeros((tile_size, tile_size), dtype=np.uint8)
for nuc in hovernet_nuclei.values():
    contour = nuc['contour']
    if len(contour) >= 3:
        rr, cc = ski_polygon(contour[:, 1], contour[:, 0], shape=(tile_size, tile_size))
        hovernet_mask[rr, cc] = 1

print(f"HoVer-Net mask nucleus pixels: {hovernet_mask.sum()}")
print(f"IDC SEG mask nucleus pixels: {seg_mask.sum()}")

## 7. Visual Comparison

In [ ]:
# Side-by-side: IDC SEG mask, HoVer-Net mask, agreement overlay
fig, axes = plt.subplots(1, 3, figsize=(21, 7))

# IDC SEG mask
idc_overlay = tile.copy()
idc_overlay[seg_mask > 0] = (idc_overlay[seg_mask > 0] * 0.5 + np.array([0, 255, 0]) * 0.5).astype(np.uint8)
axes[0].imshow(idc_overlay)
axes[0].set_title(f"IDC SEG Mask\n({seg_mask.sum()} pixels)", fontsize=13)
axes[0].axis('off')

# HoVer-Net mask
hn_overlay = tile.copy()
hn_overlay[hovernet_mask > 0] = (hn_overlay[hovernet_mask > 0] * 0.5 + np.array([255, 0, 0]) * 0.5).astype(np.uint8)
axes[1].imshow(hn_overlay)
axes[1].set_title(f"TIAToolbox HoVer-Net Mask\n({hovernet_mask.sum()} pixels)", fontsize=13)
axes[1].axis('off')

# Agreement overlay
# Green = IDC only, Red = HoVer-Net only, Yellow = both agree
agreement = tile.copy().astype(np.float32)
both = np.logical_and(seg_mask > 0, hovernet_mask > 0)
idc_only = np.logical_and(seg_mask > 0, hovernet_mask == 0)
hn_only = np.logical_and(seg_mask == 0, hovernet_mask > 0)

agreement[both] = agreement[both] * 0.4 + np.array([255, 255, 0]) * 0.6
agreement[idc_only] = agreement[idc_only] * 0.4 + np.array([0, 255, 0]) * 0.6
agreement[hn_only] = agreement[hn_only] * 0.4 + np.array([255, 0, 0]) * 0.6

axes[2].imshow(agreement.astype(np.uint8))
axes[2].set_title("Agreement Overlay", fontsize=13)
legend_patches = [
    mpatches.Patch(color='yellow', label=f'Both ({both.sum()} px)'),
    mpatches.Patch(color='green', label=f'IDC only ({idc_only.sum()} px)'),
    mpatches.Patch(color='red', label=f'HoVer-Net only ({hn_only.sum()} px)'),
]
axes[2].legend(handles=legend_patches, fontsize=9, loc='upper right')
axes[2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Zoomed-in comparison on a 512x512 region
zoom_x, zoom_y = tile_size // 4, tile_size // 4
zoom_size = 512

zoomed_tile = tile[zoom_y:zoom_y+zoom_size, zoom_x:zoom_x+zoom_size]
zoomed_seg = seg_mask[zoom_y:zoom_y+zoom_size, zoom_x:zoom_x+zoom_size]
zoomed_hn = hovernet_mask[zoom_y:zoom_y+zoom_size, zoom_x:zoom_x+zoom_size]

fig, axes = plt.subplots(1, 3, figsize=(21, 7))

# Zoomed IDC SEG
z_idc = zoomed_tile.copy()
z_idc[zoomed_seg > 0] = (z_idc[zoomed_seg > 0] * 0.5 + np.array([0, 255, 0]) * 0.5).astype(np.uint8)
axes[0].imshow(z_idc)
axes[0].set_title("IDC SEG Mask (zoomed)", fontsize=13)
axes[0].axis('off')

# Zoomed HoVer-Net with contours
axes[1].imshow(zoomed_tile)
type_colors = {1: 'red', 2: 'green', 3: 'blue', 4: 'yellow', 5: 'black'}
type_names = {1: 'Neoplastic', 2: 'Non-neoplastic', 3: 'Inflammatory', 4: 'Connective', 5: 'Dead'}
for nuc in hovernet_nuclei.values():
    centroid = nuc['centroid']
    if (zoom_x <= centroid[0] < zoom_x + zoom_size and
        zoom_y <= centroid[1] < zoom_y + zoom_size):
        contour = nuc['contour'] - np.array([zoom_x, zoom_y])
        nuc_type = nuc['type']
        color = type_colors.get(nuc_type, 'gray')
        polygon = MplPolygon(contour, closed=True, fill=True,
                             facecolor=(*plt.cm.colors.to_rgb(color), 0.3),
                             edgecolor=color, linewidth=0.8)
        axes[1].add_patch(polygon)
axes[1].set_title("TIAToolbox HoVer-Net (zoomed)", fontsize=13)
axes[1].axis('off')

# Zoomed agreement
z_agree = zoomed_tile.copy().astype(np.float32)
z_both = np.logical_and(zoomed_seg > 0, zoomed_hn > 0)
z_idc_only = np.logical_and(zoomed_seg > 0, zoomed_hn == 0)
z_hn_only = np.logical_and(zoomed_seg == 0, zoomed_hn > 0)
z_agree[z_both] = z_agree[z_both] * 0.4 + np.array([255, 255, 0]) * 0.6
z_agree[z_idc_only] = z_agree[z_idc_only] * 0.4 + np.array([0, 255, 0]) * 0.6
z_agree[z_hn_only] = z_agree[z_hn_only] * 0.4 + np.array([255, 0, 0]) * 0.6
axes[2].imshow(z_agree.astype(np.uint8))
axes[2].set_title("Agreement (zoomed)", fontsize=13)
legend_patches = [
    mpatches.Patch(color='yellow', label='Both'),
    mpatches.Patch(color='green', label='IDC only'),
    mpatches.Patch(color='red', label='HoVer-Net only'),
]
axes[2].legend(handles=legend_patches, fontsize=9, loc='upper right')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 8. Quantitative Comparison

In [ ]:
# Pixel-level segmentation metrics
intersection = np.logical_and(seg_mask > 0, hovernet_mask > 0).sum()
union = np.logical_or(seg_mask > 0, hovernet_mask > 0).sum()

dice = 2 * intersection / (seg_mask.sum() + hovernet_mask.sum()) if (seg_mask.sum() + hovernet_mask.sum()) > 0 else 0
iou = intersection / union if union > 0 else 0
precision = intersection / hovernet_mask.sum() if hovernet_mask.sum() > 0 else 0
recall = intersection / seg_mask.sum() if seg_mask.sum() > 0 else 0

print("Pixel-Level Segmentation Metrics")
print("=" * 40)
print(f"Dice coefficient:   {dice:.3f}")
print(f"IoU (Jaccard):      {iou:.3f}")
print(f"Precision:          {precision:.3f}")
print(f"Recall:             {recall:.3f}")
print(f"")
print(f"IDC SEG pixels:     {seg_mask.sum():>10,}")
print(f"HoVer-Net pixels:   {hovernet_mask.sum():>10,}")
print(f"Intersection:       {intersection:>10,}")
print(f"Union:              {union:>10,}")

In [ ]:
# Nucleus count comparison using connected components
idc_labels = ski_label(seg_mask)
hovernet_count = len(hovernet_nuclei)
idc_count = idc_labels.max()

print("Nucleus Count Comparison")
print("=" * 40)
print(f"IDC (connected components):  {idc_count:6d}")
print(f"TIAToolbox (instances):       {hovernet_count:6d}")
ratio = hovernet_count / max(idc_count, 1)
print(f"Ratio (TIAToolbox/IDC):       {ratio:.2f}")

In [ ]:
# Spatial density heatmaps — count nuclei per grid cell from binary masks
grid_size = 64  # Size of each grid cell in tile pixels
n_grid_x = tile_size // grid_size
n_grid_y = tile_size // grid_size

# Count nuclei per grid cell using connected components
idc_density = np.zeros((n_grid_y, n_grid_x))
hovernet_density = np.zeros((n_grid_y, n_grid_x))

for gy in range(n_grid_y):
    for gx in range(n_grid_x):
        y0, y1 = gy * grid_size, (gy + 1) * grid_size
        x0, x1 = gx * grid_size, (gx + 1) * grid_size
        # IDC: count connected components in this grid cell
        cell_labels = ski_label(seg_mask[y0:y1, x0:x1])
        idc_density[gy, gx] = cell_labels.max()
        # HoVer-Net: count connected components in this grid cell
        cell_labels_hn = ski_label(hovernet_mask[y0:y1, x0:x1])
        hovernet_density[gy, gx] = cell_labels_hn.max()

# Plot density heatmaps
vmax = max(idc_density.max(), hovernet_density.max())

fig, axes = plt.subplots(1, 3, figsize=(21, 6))

im0 = axes[0].imshow(idc_density, cmap='hot', vmin=0, vmax=vmax)
axes[0].set_title("IDC Nucleus Density", fontsize=13)
plt.colorbar(im0, ax=axes[0], label="Nuclei per grid cell")

im1 = axes[1].imshow(hovernet_density, cmap='hot', vmin=0, vmax=vmax)
axes[1].set_title("TIAToolbox Nucleus Density", fontsize=13)
plt.colorbar(im1, ax=axes[1], label="Nuclei per grid cell")

# Difference map
diff = hovernet_density - idc_density
vabs = max(abs(diff.min()), abs(diff.max())) if diff.any() else 1
im2 = axes[2].imshow(diff, cmap='RdBu_r', vmin=-vabs, vmax=vabs)
axes[2].set_title("Difference (TIAToolbox - IDC)", fontsize=13)
plt.colorbar(im2, ax=axes[2], label="Nucleus count difference")

plt.tight_layout()
plt.show()

In [ ]:
# Correlation between density maps
from scipy import stats

mask = (idc_density > 0) | (hovernet_density > 0)  # Non-empty cells
if mask.sum() > 2:
    corr, pval = stats.pearsonr(idc_density[mask], hovernet_density[mask])
    print(f"Spatial density correlation: r={corr:.3f}, p={pval:.2e}")

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(idc_density[mask], hovernet_density[mask], alpha=0.3, s=20)
    max_val = max(idc_density.max(), hovernet_density.max())
    ax.plot([0, max_val], [0, max_val], 'r--', alpha=0.5, label='y=x')
    ax.set_xlabel("IDC nuclei per cell", fontsize=12)
    ax.set_ylabel("TIAToolbox nuclei per cell", fontsize=12)
    ax.set_title(f"Spatial Density Correlation (r={corr:.3f})", fontsize=14)
    ax.legend()
    plt.tight_layout()
    plt.show()

## Summary

In this notebook, we learned how to:

- Discover **DICOM SEG** objects in IDC by finding segmentations in studies that contain slide microscopy (SM) data
- Use `seg_index.segmented_SeriesInstanceUID` to link each SEG to its exact source slide
- Download both the slide and its DICOM SEG binary segmentation mask
- Parse DICOM SEG objects using `highdicom` with **lazy frame retrieval** for memory-efficient access
- Extract a region's binary mask using `get_total_pixel_matrix()` without loading the entire file
- Handle the **coordinate scale factor** between the slide's baseline resolution and the SEG's pixel grid
- Run TIAToolbox's **HoVer-Net** and convert its instance segmentation to a binary mask
- **Visually compare** using agreement overlays (yellow=both, green=IDC only, red=HoVer-Net only)
- **Quantitatively compare** using pixel-level metrics (Dice, IoU, precision, recall), nucleus counts, and spatial density correlation

**Key observations:**
- The IDC SEG masks are stored as TILED_FULL BINARY segmentations with 256x256 tiles, enabling efficient region-based access via highdicom
- The SEG may be at a **different resolution** than the slide baseline (e.g., ~4x coarser for 40x slides), requiring coordinate conversion and affecting pixel-level metric precision
- Nucleus counts from connected components (IDC) and instance detection (HoVer-Net) show good agreement, confirming correct spatial alignment
- Pixel-level metrics (Dice, IoU) are sensitive to the resolution mismatch between the SEG and HoVer-Net output — these would improve on slides where the SEG resolution is closer to the analysis resolution
- TIAToolbox additionally provides nucleus **type classification** (neoplastic, inflammatory, etc.), which the binary mask does not distinguish

## Acknowledgments

- **IDC:** Fedorov, A., et al. "National Cancer Institute Imaging Data Commons: Toward Transparency, Reproducibility, and Scalability in Imaging Artificial Intelligence." *RadioGraphics* 43.12 (2023). https://doi.org/10.1148/rg.230180
- **TIAToolbox:** Pocock, J., et al. "TIAToolbox as an end-to-end library for advanced tissue image analytics." *Communications Medicine* 2, 120 (2022). https://doi.org/10.1038/s43856-022-00186-5
- **HoVer-Net:** Graham, S., et al. "Hover-Net: Simultaneous segmentation and classification of nuclei in multi-tissue histology images." *Medical Image Analysis* 58 (2019). https://doi.org/10.1016/j.media.2019.101563
- **Pan-Cancer Nuclei Segmentation:** Deng, R., et al. Pan-Cancer Nuclei Instance Segmentation and Classification. https://doi.org/10.7937/TCIA.2021.AJSK5Z43
- **highdicom:** Bridge, C., et al. "Highdicom: a Python library for standardized encoding of image annotations and machine learning model outputs in pathology and radiology." *Journal of Digital Imaging* 35 (2022). https://doi.org/10.1007/s10278-022-00683-y